<a href="https://colab.research.google.com/github/SehrishbAsghar/FlyRank_ML_Internship_Sehrish/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

In [3]:
rel = "hf://datasets/FlyRank/internship-warehouse"

content = con.sql(f"""
    SELECT content_hash_id, content_type
    FROM read_parquet('{rel}/dim_content.parquet')
""").df()

print(content.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id     content_type
0  content_004de9653278b5a4  keyword article
1  content_00dc5efae381b2ab  keyword article
2  content_01410f2556c327ac  keyword article
3  content_019f27f634053ca7  keyword article
4  content_01efa71faea45dcc  keyword article


In [5]:
rel = "hf://datasets/FlyRank/internship-warehouse"

# Load the clean slice: gsc_data_available IS TRUE, mid-panel month
raw = con.sql(f"""
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df()

# Join content_type from dim_content (single file, not a partitioned folder)
content = con.sql(f"""
    SELECT content_hash_id, content_type
    FROM read_parquet('{rel}/dim_content.parquet')
""").df()

df = raw.merge(content, on="content_hash_id", how="left")


In [6]:
df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"].replace(0, np.nan)
df["ctr"] = df["ctr"].fillna(0)

# 2. Position tier -- bucketed myself from gsc_avg_position, since the warehouse
#    ships raw position, not a pre-made tier. My bucket convention:
#    top_3 (1-3), page_1 (4-10), striking (11-20), page_3_5 (21-50), deep (51+)
def position_tier(pos):
    if pd.isna(pos):
        return "unknown"
    elif pos <= 3:
        return "top_3"
    elif pos <= 10:
        return "page_1"
    elif pos <= 20:
        return "striking"
    elif pos <= 50:
        return "page_3_5"
    else:
        return "deep"

df["position_tier"] = df["gsc_avg_position"].apply(position_tier)

# 3. Engagement rate (guard divide-by-zero, fill missing with 0 -- treated as
#    "no engaged sessions observed", not "unknown")
df["engagement_rate"] = df["ga4_engaged_sessions"] / df["ga4_sessions"].replace(0, np.nan)
df["engagement_rate"] = df["engagement_rate"].fillna(0)

# 4. Average engagement seconds per session
df["avg_engagement_sec"] = df["ga4_total_engagement_sec"] / df["ga4_sessions"].replace(0, np.nan)
df["avg_engagement_sec"] = df["avg_engagement_sec"].fillna(0)

# 5. Log-scaled impressions (raw impressions are heavily right-skewed)
df["log_impressions"] = np.log1p(df["gsc_impressions"])

# categorical handling: content_type as category dtype, unknowns filled explicitly
df["content_type"] = df["content_type"].fillna("unknown").astype("category")
df["position_tier"] = df["position_tier"].astype("category")

feature_cols = ["ctr", "position_tier", "content_type", "engagement_rate",
                 "avg_engagement_sec", "log_impressions"]

print(df[feature_cols].head(10))
print("\nMissing values per feature:")
print(df[feature_cols].isna().sum())

        ctr position_tier     content_type  engagement_rate  \
0  0.000000        page_1  keyword article              0.0   
1  0.000000         top_3  keyword article              0.0   
2  0.008000        page_1  keyword article              0.0   
3  0.000000        page_1  keyword article              0.0   
4  0.000000         top_3  keyword article              0.0   
5  0.004184        page_1  keyword article              0.0   
6  0.000000        page_1  keyword article              0.0   
7  0.000000        page_1  keyword article              0.0   
8  0.000000        page_1  keyword article              0.0   
9  0.000000        page_1  keyword article              0.0   

   avg_engagement_sec  log_impressions  
0                 0.0         3.044522  
1                 0.0         0.693147  
2                 0.0         4.836282  
3                 0.0         2.079442  
4                 0.0         2.484907  
5                 0.0         5.480639  
6                 0

In [7]:
print("Rows with ga4_sessions > 0:", (df["ga4_sessions"] > 0).sum(), "out of", len(df))
print(df["ga4_engaged_sessions"].describe())

Rows with ga4_sessions > 0: 361095 out of 3611061
count    2082695.0
mean      0.013431
std       0.134808
min            0.0
25%            0.0
50%            0.0
75%            0.0
max           21.0
Name: ga4_engaged_sessions, dtype: Float64


**Note on engagement features:**

Only 10% (app) of rows (361,095 / 3,611,061) have any GA4 session activity at all. The raw `ga4_engaged_sessions` column has nulls for about 1.5M rows (likely `ga4_data_available = FALSE` periods), which I filled to 0
during feature engineering treating "no GA4 tracking" as "0 engagement observed," a deliberate choice, not missing data left unaddressed. This means `engagement_rate` and `avg_engagement_sec` are near-zero for the large majority of rows and carry real signal only for the ~10% of rows with actual GA4 coverage.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**1. `ctr`**
- Meaning: `gsc_clicks / gsc_impressions` the observed click-through rate for this content item on this day.
- Missing: none directly (impressions/clicks had zero nulls after the `gsc_data_available IS TRUE` filter); guarded division-by-zero just in case, filled to 0.
- Available when: yes this is the day's own observed outcome, known as soon as the day's GSC data lands. It's my proxy/label input, not a feature used to predict itself.

**2. `position_tier`**
- Meaning: a bucket I defined myself from `gsc_avg_position` (top_3, page_1, striking, page_3_5, deep) the warehouse doesn't ship a pre-made tier column.
- Missing: none after the `IS TRUE` filter (all null positions came from `gsc_data_available = False` rows, already excluded).
- Available when: yes position is known the same day GSC reports it, before any engagement or click outcome is even relevant to interpret.

**3. `content_type`**
- Meaning: joined from `dim_content` the content's category (e.g. keyword article, comparison article, feedly article).
- Missing: filled to `"unknown"` for any content_hash_id that didn't match on the join (categorical, explicit bucket rather than silently dropped).
- Available when: yes, content_type is a property of the page itself, set whenever the content was created, long before any single day's performance.

**4. `engagement_rate`**
- Meaning: `ga4_engaged_sessions / ga4_sessions` share of sessions that were "engaged" per GA4's definition.
- Missing: raw column has nulls for ~1.5M rows (likely `ga4_data_available = FALSE` periods); filled to 0, treating "no GA4 tracking" as "no engagement observed" a deliberate choice. Only ~10% of rows have any real GA4 activity at all.
- Available when: yes, same-day but caution: this is itself an *outcome* of the day, not something known before a click happens. It's a feature only in the sense of "same-day context," not a true pre-decision signal if my target were "will this page get engagement" worth flagging as borderline for some framings.

**5. `avg_engagement_sec`**
- Meaning: `ga4_total_engagement_sec / ga4_sessions` average engagement duration per session.
- Missing: same handling as engagement_rate filled to 0 for no-GA4-tracking rows.
- Available when: same caveat as engagement_rate a same-day outcome measure, not a strictly pre-click signal.

**6. `log_impressions`**
- Meaning: `log1p(gsc_impressions)` log-scaled visibility volume, to reduce the effect of extreme right-skew in raw impressions.
- Missing: none (impressions had zero nulls after filtering).
- Available when: yes impressions for the day are known as soon as GSC reports, same timing as ctr and position_tier.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Build expected CTR per group, same approach as before
group_stats = df.groupby(["position_tier", "content_type"])["ctr"].transform("mean")
df["expected_ctr"] = group_stats
df["ctr_gap"] = df["ctr"] - df["expected_ctr"]

# Binary label: is this page meaningfully underperforming its group? (bottom 20% of gap)
threshold = df["ctr_gap"].quantile(0.20)
df["is_underperforming"] = (df["ctr_gap"] <= threshold).astype(int)

# Honest feature set (no leakage)
honest_features = ["log_impressions", "engagement_rate", "avg_engagement_sec"]
X = df[honest_features].fillna(0)
y = df["is_underperforming"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

clf = DecisionTreeClassifier(max_depth=4, random_state=42)
clf.fit(X_train, y_train)
honest_score = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])
print(f"Honest ROC AUC (no leakage): {honest_score:.4f}")

/tmp/ipykernel_1509/1477250893.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_stats = df.groupby(["position_tier", "content_type"])["ctr"].transform("mean")


Honest ROC AUC (no leakage): 0.5812


In [9]:
# --- THE TRAP: add a label-derived column on purpose
X_leaky = df[honest_features + ["ctr_gap"]].fillna(0)  # ctr_gap literally defines the label

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaky, y, test_size=0.3, random_state=42, stratify=y)

clf_leaky = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_leaky.fit(X_train_l, y_train_l)
leaky_score = roc_auc_score(y_test_l, clf_leaky.predict_proba(X_test_l)[:, 1])
print(f"Leaky ROC AUC (with ctr_gap as a feature): {leaky_score:.4f}")

Leaky ROC AUC (with ctr_gap as a feature): 1.0000


**The trap:** `is_underperforming` was defined by thresholding `ctr_gap` (the bottom 20% of the gap distribution).

I deliberately added `ctr_gap` itself back in as a feature to predict that same label.

**Honest ROC AUC (no leakage):**

0.5812, built from genuinely pre-decision features only (`log_impressions`, `engagement_rate`, `avg_engagement_sec`).

Modest, realistic signal nowhere near perfect, which is exactly what real, independent features should look like.

**Leaky ROC AUC (ctr_gap included):**

1.0000, a perfect score, and a red flag, not a win. Since the label was literally thresholded from `ctr_gap`, giving the model that column let it "predict" by reading back its own answer a circular
result, not a discovery.

**Fix:** delete `ctr_gap` (and any other column directly derived from the label) from the feature set. The honest number to report and build on is 0.5812, not
1.0000. This mirrors the same lesson from notebook 02 (the depth-4 decision tree's in-sample vs. held-out precision comparison) here, performed on real warehouse
data, with a label I defined myself, not just a starter-CSV exercise.

In [10]:
print(f"Reverting to honest feature set (no ctr_gap): ROC AUC = {honest_score:.4f}")
print("This is the number that gets reported and built on going forward.")

Reverting to honest feature set (no ctr_gap): ROC AUC = 0.5812
This is the number that gets reported and built on going forward.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- **`sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`,
  `sessions_paid`, `sessions_ai`**
, traffic-channel breakdowns.

Excluded because my lane is CTR/ Engagement Opportunity Scoring from search, not channel attribution; mixing these in would blur what's actually driving the score.

- **`ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other`**

AI-referral breakdowns, belonging to the separate "AI Referral Opportunity" freestyle direction, not this lane.

- **`scroll_events`** genuinely relevant per the lane guide, but excluded from this feature for now because it's extremely sparse (only 600,821 of 78.8M total warehouse rows have any scroll data at all, per the lane guide's
metric-density table) including it here would mean filling ~99% of rows with a fabricated zero, which risks the model treating "no data" as "true zero scroll behavior."

- **`gsc_sum_position`** — redundant with `gsc_avg_position` (sum is just avg x count); no new information, just noise collinearity risk.

- **`ctr_gap` (and `expected_ctr`)** deliberately excluded from the final honest feature set, since they're derived directly from the label (`is_underperforming`) itself. Keeping them in would be the exact leakage trap demonstrated in Section 3, they're diagnostic/label-construction fields, not legitimate predictive features.

- **Any product-computed flag(e.g. `health_score`, `priority_score`,
  `action_type`)**

not applicable here since the warehouse release doesn't ship
these at all (per the lane guide, FlyRank's product decision flags are deliberately excluded from the data). Noting their absence explicitly so it's clear this wasn't overlooked there was nothing to strip out.

- **`client_hash_id`, `content_hash_id`, `report_date`, `month`**

identifiers and time-slicing fields, not predictive signal. Used only for joins, filtering, and grouping, never as model inputs.

## Self-check

Before you submit, confirm each line honestly:

- Every section above is filled — markdown thinking AND the code that backs it
- The notebook runs top to bottom with no errors (Runtime → Run all)
- No client names, URLs, or private queries anywhere
- My claims use careful words: observed, measured, directional, decision-support
- Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.